In [2]:
import random
from datetime import datetime
from datetime import timedelta

import pandas as pd
from faker import Faker

In [3]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])

In [4]:
def generate_fake_data(n: int):
    return [{
        "id": i + 1,
        "name": fake.name(),
        "age": fake.random_int(min=20, max=65),
        "weight": fake.random_int(min=10, max=250),
        "height": fake.random_int(min=10, max=250),
        "gender": random.choice(["남성", "여성"]),
        "address": fake.address(),
        "job": fake.job(),
        "email": fake.email(),
        "signup": (datetime.now() - timedelta(days=random.randint(0, 365))).date().isoformat()
    } for i in range(n)]

### Pandas Display 설정

In [5]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 200)
pd.set_option("display.expand_frame_repr", False)

### Pandas DataFrame 만들기

In [6]:
source_dataframe = pd.DataFrame(generate_fake_data(100))
source_dataframe.head(1)

,id,name,age,weight,height,gender,address,job,email,signup
0,1,이상현,63,161,18,남성,인천광역시 성북구 잠실길 지하430,중식 주방장 및 조리사,blakeerik@example.com,2025-10-26


In [7]:
column = ["id", "name", "age", "height", "weight", "gender", "address", "job", "email", "signup"]
source_dataframe = pd.DataFrame(generate_fake_data(100), columns=column)
source_dataframe.head(1)

,id,name,age,height,weight,gender,address,job,email,signup
0,1,Christopher Flores,21,145,24,여성,"13348 Kathryn Forges\nKellyburgh, HI 31506",Mechanical engineer,hayun92@example.net,2025-05-30


### Pandas DataFrame 파일 쓰기

In [8]:
source_dataframe.to_csv("./data/customers.csv", index=False, header=False, sep="|")
source_dataframe.to_json("./data/customers.json", orient="records")
source_dataframe.to_pickle("./data/customers.pickle")
source_dataframe.to_parquet("./data/customers.parquet")

### Pandas DataFrame 파일 읽기

In [9]:
dataframe = pd.read_csv("./data/customers.csv", header=None, sep="|", names=column)
dataframe = pd.read_json("./data/customers.json")
dataframe = pd.read_parquet("./data/customers.parquet")
dataframe = pd.read_pickle("./data/customers.pickle")

In [10]:
pd.set_option("display.max_colwidth", None)
dataframe[["address", "email"]].head()
dataframe.head(1)

,id,name,age,height,weight,gender,address,job,email,signup
0,1,Christopher Flores,21,145,24,여성,"13348 Kathryn Forges\nKellyburgh, HI 31506",Mechanical engineer,hayun92@example.net,2025-05-30


### S3 (minio) 데이터 저장하기

In [11]:
storage_options = {"client_kwargs": {"endpoint_url": "http://minio:9000"}}
customer = pd.DataFrame(generate_fake_data(100))
customer.to_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers.json", orient="records", index=False, storage_options=storage_options)
customer.to_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers.json.zstd", orient="records", compression="zstd", index=False, storage_options=storage_options)
customer.to_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers_records_line.json", orient="records", index=False, lines=True, storage_options=storage_options)
customer.to_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers_index.json", orient="index", storage_options=storage_options)
customer.to_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers_split.json", orient="split", storage_options=storage_options)
customer.to_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers_values.json", orient="values", storage_options=storage_options)
customer.to_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers_table.json", orient="table", storage_options=storage_options)

### S3 (minio) 데이터 불러오기

In [12]:
customer = pd.read_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers.json", storage_options=storage_options)
customer_zstd = pd.read_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers.json.zstd", compression="zstd", storage_options=storage_options)
customers_records_line = pd.read_json("s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customers_records_line.json", lines=True, storage_options=storage_options)

In [13]:
customers_records_line

,id,name,age,weight,height,gender,address,job,email,signup
0,1,문영미,24,159,23,남성,"65273 Cantrell Flats Suite 209\nWest Derekmouth, NH 55318",재활용 처리 및 소각로 조작원,oliviamunoz@example.net,2025-09-01
1,2,최서윤,53,230,161,여성,"63356 Patricia Ramp Apt. 906\nNorth Vickiehaven, AZ 56298",Quality manager,fbag@example.net,2025-07-08
2,3,이서윤,38,175,14,남성,"799 Tiffany Mountain\nPort Jacqueline, HI 49052",Accommodation manager,vson@example.net,2025-10-06
3,4,권정식,63,168,198,여성,서울특별시 성동구 영동대90거리 지하180 (보람황읍),Field trials officer,wseo@example.net,2025-12-06
4,5,권숙자,52,80,26,남성,"817 Hayes Land Apt. 371\nSouth Rachaelport, CA 14835",부동산 컨설턴트 및 중개인,myeongjayang@example.net,2025-02-05
5,6,주영철,50,117,198,여성,인천광역시 용산구 언주거리 352 (경희장박마을),"Programmer, systems",rachelparker@example.com,2026-01-15
6,7,주종수,29,82,52,남성,대구광역시 동대문구 역삼길 325 (진호이읍),표백 및 염색 관련 조작원,earl78@example.org,2025-06-02
7,8,Alexander Velazquez,49,49,199,여성,"4929 Morgan Creek Suite 028\nWest Angelbury, NV 47345",실내장식 디자이너,seoyungim@example.com,2025-10-22
8,9,박시우,52,59,91,여성,울산광역시 광진구 영동대50거리 316 (지연김서리),Probation officer,lindawaller@example.com,2025-10-26
9,10,Kevin Miller,39,232,26,남성,"462 Page Throughway Apt. 326\nAngelaton, GU 40083","Research officer, political party",anthonysmith@example.com,2025-10-01


### Pandas Apply 맛보기

In [14]:
def calc_bmi(row):
    height_m = row["height"] / 100
    return round(row["weight"] / (height_m ** 2), 2)


dataframe["bmi"] = dataframe.apply(calc_bmi, axis=1)
dataframe.head(1)

,id,name,age,height,weight,gender,address,job,email,signup,bmi
0,1,Christopher Flores,21,145,24,여성,"13348 Kathryn Forges\nKellyburgh, HI 31506",Mechanical engineer,hayun92@example.net,2025-05-30,11.41


In [15]:
def bmi_level(row):
    if row["bmi"] < 18.5: return "저체중"
    if row["bmi"] < 23: return "정상"
    if row["bmi"] < 25: return "과체중"
    return "비만"


dataframe["bmi_level"] = dataframe.apply(bmi_level, axis=1)
dataframe.head(1)

,id,name,age,height,weight,gender,address,job,email,signup,bmi,bmi_level
0,1,Christopher Flores,21,145,24,여성,"13348 Kathryn Forges\nKellyburgh, HI 31506",Mechanical engineer,hayun92@example.net,2025-05-30,11.41,저체중


In [16]:
# Column 기준 apply (axis=0)
dataframe.apply(lambda s: s.isna().mean(), axis=0)

id           0.0
name         0.0
age          0.0
height       0.0
weight       0.0
gender       0.0
address      0.0
job          0.0
email        0.0
signup       0.0
bmi          0.0
bmi_level    0.0
dtype: float64

### Pandas Group 맛보기

In [17]:
dataframe.groupby("gender")[["age", "height", "weight"]].mean()

# 성별 인원 수 (count)
dataframe.groupby("gender").size().reset_index(name="count")
dataframe["gender"].value_counts().reset_index(name="count")

# 직업별 평균 나이 + 인원 수
dataframe.groupby("job").agg(avg_age=("age", "mean"), cnt=("id", "count")).reset_index()

# 성별 + 직업별 평균 나이 (다중 그룹)
dataframe.groupby(["gender", "job"])["age"].mean().reset_index()

# 가입 연도별 인원 수 (날짜 파생)
dataframe["signup"] = pd.to_datetime(dataframe["signup"])
dataframe["signup_year"] = dataframe["signup"].dt.year
dataframe.groupby("signup_year").size().reset_index(name="count")

# 연도 + 성별 가입자 수
dataframe.groupby([dataframe["signup"].dt.year, "gender"]).size().reset_index(name="count")

# 성별 나이 분포 (bucket)
bins = [20, 30, 40, 50, 60, 70]
dataframe["age_group"] = pd.cut(dataframe["age"], bins=bins, right=False)
dataframe.groupby(["gender", "age_group"], observed=True).size().reset_index(name="cnt")

# 성별 상위 5개 직업 (인원 기준)
dataframe.groupby(["gender", "job"]).size().reset_index(name="cnt").sort_values(["gender", "cnt"], ascending=[True, False]).groupby("gender").head(5)

# groupby 결과를 dict로 (API 응답용)
dataframe.groupby("gender")[["age", "height"]].mean().to_dict()


{'age': {'남성': 40.74545454545454, '여성': 41.37777777777778},
 'height': {'남성': 132.2, '여성': 133.15555555555557}}